In [2]:
import Pkg
Pkg.add("HiGHS")
Pkg.add("JuMP")

    Updating registry at `C:\Users\huste\.julia\registries\General.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Project.toml`
    Manifest No packages added to or removed from `C:\Users\huste\.julia\environments\v1.12\Manifest.toml`


In [3]:


using JuMP, HiGHS

########## ---------- Variables ---------- ##########
coords = [
    0 0 0;
    1 104 19;
    2 370 305;
    3 651 221;
    4 112 121;
    6 134 515
]

n_stops = size(coords, 1)

dist = zeros(Float64, n_stops, n_stops)

for i in 1:n_stops
    for j in 1:n_stops
        x1 = coords[i, 2]
        y1 = coords[i, 3]

        x2 = coords[j, 2]
        y2 = coords[j, 3]

        dist[i, j] = sqrt((x1 - x2)^2 + (y1 - y2)^2)
    end
end


########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- Variables ---------- ##########
@variable(model, x[1:n_stops, 1:n_stops], Bin)
@variable(model, u[1:n_stops] >= 1)



########## ---------- Objectives ---------- ##########
@objective(model, Min, sum(x[i,j] * dist[i,j] for i in 1:n_stops, j in 1:n_stops))

########## ---------- Constraints ---------- ##########
@constraint(model, [i in 1:n_stops], x[i, i] == 0)
@constraint(model, [i in 1:n_stops], sum(x[i,j] for j in 1:n_stops) == 1)
@constraint(model, [j in 1:n_stops], sum(x[i,j] for i in 1:n_stops) == 1)

########## ---------- Subtour elimination (MTZ) ---------- ##########
# This is the order constraint
@constraint(model, u[1] == 1)
@constraint(model, [i in 2:n_stops], u[i] >= 2)
@constraint(model, [i in 2:n_stops], u[i] <= n_stops)

# For i != j and i,j in 2..n:
# u[i] - u[j] + n*x[i,j] <= n-1
@constraint(model, [i in 2:n_stops, j in 2:n_stops; i != j],
    u[i] - u[j] + n_stops * x[i, j] <= n_stops - 1
)

optimize!(model)
println("Optimal solution:")
println(objective_value(model))
println(value.(u))

for i in 1:n_stops
    println(value.(x[i, :]))
end

Optimal solution:
1857.5117454357478
[1.0, 5.999999999999993, 3.9999999999999964, 4.9999999999999964, 1.9999999999999996, 2.9999999999999982]
[0.0, -0.0, 0.0, 0.0, 0.9999999999999996, 0.0]
[1.0, 0.0, 0.0, -0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 1.0, 0.0, -0.0]
[0.0, 0.9999999999999996, -0.0, 0.0, -0.0, -0.0]
[0.0, -0.0, -0.0, -0.0, 0.0, 1.0]
[-0.0, 0.0, 1.0, 0.0, -0.0, 0.0]


In [6]:
#************************************************************************
# Wehicle Routing, "Mathematical Programming Modelling" (42112)
using JuMP
using HiGHS
#************************************************************************
#************************************************************************
# PARAMETERS
C=13 # no of customers
# customer coordinates and store:
coord = zeros(C,2)
coord[1,1]=0; coord[1,2]=0
coord[2,1]=104; coord[2,2]=19
coord[3,1]=370; coord[3,2]=305
coord[4,1]=651; coord[4,2]=221
8
coord[5,1]=112; coord[5,2]=121
coord[6,1]=134; coord[6,2]=515
coord[7,1]=797; coord[7,2]=424;
coord[8,1]=347; coord[8,2]=444;
coord[9,1]=756; coord[9,2]=141;
coord[10,1]=304; coord[10,2]=351;
coord[11,1]=236; coord[11,2]=775;
coord[12,1]=687; coord[12,2]=310;
coord[13,1]=452; coord[13,2]=57;
# distance between customers
Distance = zeros(Float64,C,C)
for c1 in 1:C
for c2 in 1:C
Distance[c1,c2]= sqrt( (coord[c1,1]-coord[c2,1])^2 + (coord[c1,2]-coord[c2,2])^2)
end
end
# size of deliveries
q = zeros(C)
q[1]=0;
q[2]=3;
q[3]=9;
q[4]=7;
q[5]=11;
q[6]=11;
q[7]=6;
q[8]=7;
q[9]=7;
q[10]=2;
q[11]=4;
q[12]=2;
q[13]=8;
# number of vans
K=3
# capacity of each of the vans
Q=26
#************************************************************************
#************************************************************************
# Model
VRP = Model(HiGHS.Optimizer)
@variable(VRP, x[1:C,1:C],Bin)
9
for c in 1:C
fix(x[c,c],0; force = true) #"Remove" nonsense variables by setting to 0
end
@variable(VRP, 0 <= u[1:C] <= Q)
# Minimize VRP distance
@objective(VRP, Min,
sum(Distance[c1,c2]*x[c1,c2] for c1=1:C, c2=1:C))
# every customer is entered by one van, except the store on node 1
@constraint(VRP, [c1=2:C],
sum(x[c2,c1] for c2=1:C) == 1)
# every customer is exited by one van, except the store on node 1
@constraint(VRP, [c1=2:C],
sum(x[c1,c2] for c2=1:C) == 1)
# K vans going out of node 1
@constraint(VRP,
sum(x[1,c2] for c2=2:C) == K)
# K trucks going in of node 1
@constraint(VRP,
sum(x[c2,1] for c2=2:C) == K)
# Combined counter and capacity constraint
@constraint(VRP, [c1=2:C,c2=1:C,c1!=c2],
u[c1] + q[c1] <= u[c2] + Q*(1-x[c1,c2]) )
#************************************************************************
#************************************************************************
# solve
optimize!(VRP)
println("Termination status: $(termination_status(VRP))")
#************************************************************************
#************************************************************************
#print(VRP)
println("objective = $(objective_value(VRP))")
println("Solve time: $(solve_time(VRP))")
#************************************************************************

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 182 rows; 182 cols; 780 nonzeros; 169 integer variables (156 binary)
Coefficient ranges:
  Matrix  [1e+00, 3e+01]
  Cost    [8e+01, 9e+02]
  Bound   [1e+00, 3e+01]
  RHS     [1e+00, 2e+01]
Presolving model
170 rows, 168 cols, 732 nonzeros  0s
170 rows, 168 cols, 732 nonzeros  0s
Presolve reductions: rows 170(-12); columns 168(-14); nonzeros 732(-48) 

Solving MIP model with:
   170 rows
   168 cols (156 binary, 0 integer, 0 implied int., 12 continuous, 0 domain fixed)
   732 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tre

# Tennis chours

In [ ]:
import Pkg
Pkg.add("HiGHS")
Pkg.add("JuMP")

using JuMP, HiGHS


In [ ]:
coords = [
    1   0.0   0.0;    # A0
    2   4.5   0.0;    # B0
    3  31.5   0.0;    # D0
    4  36.0   0.0;    # E0
    5   4.5  18.0;    # B1
    6  18.0  18.0;    # C1
    7  31.5  18.0;    # D1
    8   0.0  39.0;    # A2
    9   4.5  39.0;    # B2
   10  18.0  39.0;    # C2
   11  31.5  39.0;    # D2
   12  36.0  39.0     # E2
]

n_stops = size(coords)[1]
possible_routes = zeros(Int, 12, 12)

edges = [
    (1,2), (1,8),
    (2,5), (2,3),
    (3,4), (3,7),
    (4,12),
    (5,6), (5,9),
    (6,7), (6,10),
    (7,11)
]

for (i,j) in edges
    possible_routes[i,j] = 1
    possible_routes[j,i] = 1   # remove this line if you want directed
end


distance = zeros(Float64, n_stops, n_stops)

for i in 1:n_stops
    for j in 1:n_stops
        if possible_routes[i,j] == 1
            x1 = coords[i, 2]
            y1 = coords[i, 3]

            x2 = coords[j, 2]
            y2 = coords[j, 3]

            distance[i, j] = sqrt((x1 - x2)^2 + (y1 - y2)^2)
        end
    end
end



########## ---------- Models ---------- ##########
########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- Variables ---------- ##########    possible_routes[j,i] = 1   # remove this line if you want directed

@variable(model, x[1:n_stops, 1:n_stops], Bin)



########## ---------- Objectives ---------- ##########
@objective(model, Min, sum(x[i,j] * distance[i,j] for i in 1:n_stops, j in 1:n_stops))

########## ---------- Constraints ---------- ##########
# Only allow traversal on permitted arcs
for i in 1:n_stops, j in 1:n_stops
    if possible_routes[i,j] == 0
        @constraint(model, x[i,j] == 0)
    end
end

# Flow conservation (what goes in must come out)
@constraint(model, [i in 1:n_stops],
    sum(x[i,j] for j in 1:n_stops) == sum(x[j,i] for j in 1:n_stops)
)

# Force sweeping of required undirected lines
@constraint(model, [i in 1:n_stops, j in 1:n_stops],
    x[i,j] + x[j,i] >= possible_routes[i,j]
)


optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))



Optimal solution:
z = 300.0
